# Bài tập Logistic Regression

File `Social_Network_Ads.csv` nằm cùng thư mục với notebook trong `Practice/`.

Mỗi dòng mô tả một khách hàng:
- `Age` và `EstimatedSalary`: đặc trưng đầu vào (features).
- `Purchased`: nhãn cần dự đoán; `0` là không mua, `1` là có mua.

Mục tiêu là dự đoán khách hàng có mua sản phẩm hay không.

## Bước 1: Tải dữ liệu

Đọc file CSV vào bảng `data` và xem 5 dòng đầu tiên để kiểm tra tên cột cùng dữ liệu mẫu.

In [1]:
from pathlib import Path
import pandas as pd

# Tìm file CSV trong thư mục hiện tại hoặc thư mục Practice.
csv_path = Path("Social_Network_Ads.csv")
if not csv_path.exists():
    csv_path = Path("Practice") / "Social_Network_Ads.csv"

# Đọc dữ liệu và xem nhanh 5 dòng đầu.
data = pd.read_csv(csv_path)
data.head()

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0


## Bước 2: Tách dữ liệu và chia tập train/test

- `features` gồm hai cột `Age` và `EstimatedSalary`.
- `label` là cột `Purchased`.
- Dùng 80% dữ liệu để huấn luyện và 20% để kiểm tra. `random_state=0` giúp kết quả chia lặp lại giống nhau.

In [2]:
from sklearn.model_selection import train_test_split

# Hai đặc trưng đầu vào mà mô hình dùng để dự đoán.
features = data[["Age", "EstimatedSalary"]]

# Nhãn cần dự đoán: 0 là không mua, 1 là có mua.
label = data["Purchased"]

# Dành 80% dữ liệu để học và 20% để kiểm tra kết quả.
x_train, x_test, y_train, y_test = train_test_split(
    features, label, test_size=0.2, random_state=0
)

## Bước 3: Huấn luyện mô hình

Tạo `LogisticRegression(max_iter=100)` rồi huấn luyện bằng tập train.

- `intercept_` là bias (hệ số chặn).
- `coef_[0][0]` là trọng số của `Age`.
- `coef_[0][1]` là trọng số của `EstimatedSalary`.
- `n_iter_[0]` cho biết mô hình đã chạy bao nhiêu vòng lặp.

Công thức tính điểm: `z = bias + weight_age * Age + weight_salary * EstimatedSalary`.

In [3]:
from sklearn.linear_model import LogisticRegression

# Tạo mô hình và huấn luyện bằng dữ liệu train.
model = LogisticRegression(max_iter=100)
model.fit(x_train, y_train)

# Lấy bias và trọng số theo đúng thứ tự cột: Age, EstimatedSalary.
bias = model.intercept_[0]
weight_age = model.coef_[0][0]
weight_salary = model.coef_[0][1]

print(f"Số vòng lặp: {model.n_iter_[0]}")
print(f"Bias: {bias:.6f}")
print(f"Trọng số Age: {weight_age:.6f}")
print(f"Trọng số EstimatedSalary: {weight_salary:.6f}")
print(f"z = {bias:.6f} + {weight_age:.6f} * Age + {weight_salary:.6f} * EstimatedSalary")

Số vòng lặp: 58
Bias: -11.297709
Trọng số Age: 0.209982
Trọng số EstimatedSalary: 0.000034
z = -11.297709 + 0.209982 * Age + 0.000034 * EstimatedSalary


## Bước 4: Dự đoán cho một khách hàng

Dự đoán khách hàng 48 tuổi, lương 50,000. `predict_proba` trả về xác suất cho từng lớp; lấy `[0][1]` để lấy xác suất thuộc lớp `1` (có mua).

Nếu xác suất từ `0.5` trở lên, dự đoán là “Mua”; nếu thấp hơn, dự đoán là “Không mua”. Cuối cùng, xem độ chính xác của mô hình trên tập test.

In [4]:
# Tạo bảng 2 chiều, giữ đúng thứ tự và tên hai cột đã dùng để huấn luyện.
customer = pd.DataFrame({"Age": [48], "EstimatedSalary": [50000]})

# Lấy xác suất thuộc lớp 1 (có mua).
probability = model.predict_proba(customer)[0][1]

# Dùng ngưỡng 0.5 để đổi xác suất thành nhãn dự đoán.
prediction = 1 if probability >= 0.5 else 0
result = "Mua" if prediction == 1 else "Không mua"

# Độ chính xác được tính trên tập test mà mô hình chưa dùng để học.
accuracy = model.score(x_test, y_test)

print(f"Xác suất mua: {probability:.2%}")
print(f"Dự đoán: {result}")
print(f"Độ chính xác trên tập test: {accuracy:.2%}")

Xác suất mua: 61.55%
Dự đoán: Mua
Độ chính xác trên tập test: 91.25%
